In [3]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM
import json
from helpers.text_refiners import refine_caption
from helpers.video_helpers import forward_and_decode

/home/aash/miniconda3/envs/captioningenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_name = 'google/gemma-4-E2B-it'
model = AutoModelForMultimodalLM.from_pretrained( # Changed from AutoModelForVision2Seq
            model_name,
            device_map="auto" # Automatically map model to available devices (GPU if present)
        )
processor = AutoProcessor.from_pretrained(model_name)

Loading weights: 100%|██████████| 1951/1951 [00:18<00:00, 104.49it/s]


In [5]:
instruction = " Refine the text captions provided by the VLM, include the key details mentioned by the models, remove the repeatign text and make the overall captions concise"

In [6]:
with open('whole_video/tvsum_complete_qwen.json','r') as f:
    captions = json.load(f)

In [7]:
with open('per_annotator/tvsum/per_annotator.json','r') as f:
    captions_multi = json.load(f)

In [8]:
example_caption = captions['video_50']


In [24]:
example_caption_multi = captions_multi['video_22']['Annotator_2']


In [25]:
example_caption_multi

'This video is a montage showcasing a parkour event. It begins with individuals practicing movements, followed by shots of participants wearing "Charlotte Parkour" apparel. The video then transitions to outdoor parkour challenges, including vaulting over structures, performing a flip, and various group activities. The footage concludes with a group posing on a brick wall and a final title card displaying "CVP Charlotte Video Project."'

In [30]:
def refine_caption(processor,generated_captions,processor_kwargs= {}):
  new_message = f"""You are provided with automatically generated captions from a video.
  Please do the following:
  Refine the cations: Remove unnecessary formatting details, such as **title cards**.
  Keep important and relevant details, especially details unique to the description, such as important people, places logos, etc.
  Remove repeating information and timestamps/timecodes(i.e 00:00,)
    Keep the final captions concise,reducing each sentence to be 15 words long
  Rephrase sentences like: "The video shows" into something that describes the scene
  Return only the rewritten captions
  Here are the captions you must rewrite: {generated_captions}"""
  content = [{"type":"text","text":new_message}]
  batch_message = [{"role":"user","content":content}]
  messages = processor.apply_chat_template(batch_message,
                                          tokenize=True,
                                          return_dict=True,
                                          return_tensors="pt",
                                          padding=True,
                                          add_generation_prompt=True, **processor_kwargs)
  input_len = messages["input_ids"].shape[-1]
  return messages,input_len

In [31]:
messages,input_lens = refine_caption(processor,example_caption_multi)

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


In [32]:
with torch.inference_mode():
    outputs = forward_and_decode(model,messages,input_lens,processor,128,True)

In [33]:
outputs

'Individuals practice parkour movements in an exciting montage. Participants showcase "Charlotte Parkour" apparel during outdoor challenges. The footage includes vaulting, flipping, and group activities in the park. A group poses on a brick wall concluding the parkour event.'